<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-MCD8016/blob/main/Session3_0_ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

# ResNet-34

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self,in_chan,out_chan):
        super(BasicBlock,self).__init__()

        # Si hay cambio en el numero de canales, reducimos las dimensiones espaciales entre dos
        projection = not (in_chan==out_chan)
        stride = 2 if projection else 1

        self.conv1 = nn.Conv2d(in_chan,out_chan,kernel_size=3,stride=stride,
                               padding=1,bias=False)
        self.bn1   = nn.BatchNorm2d(out_chan)
        # Desactivar el bias si aplicas alguna normalizacionj

        self.conv2 = nn.Conv2d(out_chan,out_chan,kernel_size=3,stride=1,
                               padding=1,bias=False)
        self.bn2   = nn.BatchNorm2d(out_chan)

        if projection:  # Hay reduccion de dimensiones espaciales y aumento de dimensiones de canales
            self.proj = nn.Sequential(
                # Aumentamos el numero de canales y reducimos las dimensiones espaciales
                nn.Conv2d(in_chan,out_chan,kernel_size=1,stride=stride,padding=1,bias=False)
                nn.BatchNorm2d(out_chan)
            )
        else:
            self.proj = nn.Identity()

        self.ReLU = nn.ReLU(inplace=True)

    def forward(self,x):
        # Flujo principal
        z = self.conv1(x)
        z = self.bn1  (z)
        z = self.ReLU (z)

        z = self.conv2(z)
        z = self.bn2  (z)

        # Projection
        x = self.proj(x)

        return self.ReLU(x+z)

In [ ]:
class ResNet34(nn.Module):
    def __init__(self,n_classes):
        super(ResNet34,self).__init__()

        self.layer0 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        self.layer1 = nn.Sequential(
            BasicBlock(64,64),
            BasicBlock(64,64),
            BasicBlock(64,64),
        )
        self.layer2 = nn.Sequential(
            BasicBlock( 64,128),
            BasicBlock(128,128),
            BasicBlock(128,128),
            BasicBlock(128,128),
        )
        self.layer3 = nn.Sequential(
            BasicBlock(128,256),
            BasicBlock(256,256),
            BasicBlock(256,256),
            BasicBlock(256,256),
            BasicBlock(256,256),
            BasicBlock(256,256),
        )
        self.layer4 = nn.Sequential(
            BasicBlock(256,512),
            BasicBlock(512,512),
            BasicBlock(512,512),
        )

        self.AvgPool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512,n_classes)

    def forward(self,x):
        # x: [n_batch,n_chan,h,w]
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)  # [n_batch,n_chan,h,w]
                            #        0      1 2 3
        x = self.AvgPool(x) # [n_batch,n_chan,1,1]
        x = self.fc(x)

        return x

In [ ]:
net = ResNet34(n_classes=1000)
x = torch.randn(2, 3, 224, 224)
y = net(x)
print(y.shape)

torch.Size([2, 1000])


In [ ]:
print(net)

ResNet34(
  (layer0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (proj): Identity()
      (ReLU): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running

# ResNet-50

In [ ]:
class Bottleneck(nn.Module):
    def __init__(self, in_chan, out_chan):
        super().__init__()
        projection = (in_chan != out_chan)
        stride = 2 if projection else 1
        mid_chan = out_chan // 4

        self.conv1 = nn.Conv2d(in_chan,mid_chan,kernel_size=1,padding=1,stride=1,bias=False)
        self.bn1   = nn.BatchNorm2d(mid_chan)

        self.conv2 = nn.Conv2d(mid_chan,mid_chan,kernel_size=3,padding=1,stride=stride,bias=False)
        self.bn2   = nn.BatchNorm2d(mid_chan)

        self.conv3 = nn.Conv2d(mid_chan,out_chan,kernel_size=1,padding=1,stride=1,bias=False)
        self.bn3   = nn.BatchNorm2d(out_chan)

        if self.proj = nn.Sequential(
                # Aumentamos el numero de canales y reducimos las dimensiones espaciales
                nn.Conv2d(in_chan,out_chan,kernel_size=1,stride=2,bias=False),
                nn.BatchNorm2d(out_chan)
            )
        else:
            self.proj = nn.Identity()

        self.ReLU = nn.ReLU(inplace=True)

    def forward(self,x):
        # Flujo principal

        # Comprimen (1x1 conv)
        z = self.conv1(x)
        z = self. bn1(z)
        z = self.ReLU(z)

        # Procesan (3x3 conv)
        z = self.conv2(z)
        z = self. bn2(z)
        z = self.ReLU(z)

        # Expanden (1x1 conv)
        z = self.conv3(z)
        z = self. bn3(z)

        # Projection
        x = self.proj(x)

        return self.ReLU(x+z)


In [ ]:
class ResNet50(nn.Module):
    def __init__(self,n_classes):
        super(ResNet50,self).__init__()

        self.layer0 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = nn.Sequential(
            Bottleneck(64,  256),
            Bottleneck(256, 256),
            Bottleneck(256, 256),
        )
        self.layer2 = nn.Sequential(
            Bottleneck(256, 512),
            Bottleneck(512, 512),
            Bottleneck(512, 512),
            Bottleneck(512, 512),
        )
        self.layer3 = nn.Sequential(
            Bottleneck(512, 1024),
            Bottleneck(1024, 1024),
            Bottleneck(1024, 1024),
            Bottleneck(1024, 1024),
            Bottleneck(1024, 1024),
            Bottleneck(1024, 1024),
        )
        self.layer4 = nn.Sequential(
            Bottleneck(1024, 2048),
            Bottleneck(2048, 2048),
            Bottleneck(2048, 2048),
        )

        self.AvgPool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(2048, n_classes)

    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.AvgPool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
net = ResNet50(n_classes=1000)
x = torch.randn(2, 3, 224, 224)
y = net(x)
print(y.shape)

torch.Size([2, 1000])


In [ ]:
print(net)

ResNet50(
  (layer0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (proj): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(2, 2), bias=False)
